# Haptic Signal VAE - Train, Freeze, and Export

Minimal Colab workflow for the first stage only:

1. Train the VAE using the same setup and config logic as `train_colab.ipynb`.
2. Freeze the trained VAE checkpoint as the source model for later analysis.
3. Extract deterministic latent `mu`, fit PCA, and save PCA artifacts.
4. Build the basic controllable PC specification.
5. Copy the model, PCA, and controls artifacts to Google Drive.

This notebook intentionally omits extended validation, AE baseline, cross-seed comparison, decode demos, and visualization galleries.

In [ ]:
# 1. Clone code repo (or force-update to latest)
import os

REPO_URL = "https://github.com/cindy-77jiayi/thesis_hapticAE.git"
REPO_DIR = "/content/thesis_hapticAE"

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git fetch origin && git reset --hard origin/main
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!git log --oneline -3
print(f"\nWorking directory: {os.getcwd()}")

In [ ]:
# 2. Clone dataset
DATASET_URL = "https://github.com/HapticGen/hapticgen-dataset.git"
DATASET_DIR = "/content/hapticgen-dataset"

if os.path.exists(DATASET_DIR):
    !cd {DATASET_DIR} && git pull
else:
    !git clone {DATASET_URL} {DATASET_DIR}

DATA_DIR = DATASET_DIR  # scan both expertvoted/ and uservoted/
print(f"Dataset root: {DATA_DIR}")
for sub in ["expertvoted", "uservoted"]:
    p = os.path.join(DATA_DIR, sub)
    n = len(os.listdir(p)) if os.path.exists(p) else 0
    print(f"  {sub}: {n} files")

In [ ]:
# 3. Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# 4. Configure run profile (aligned with current repo configs)
OUTPUT_DIR = "/content/outputs"

# Options: "baseline_balanced", "baseline_mixed"
RUN_PROFILE = "baseline_balanced"

PROFILE_TO_CONFIG = {
    "baseline_balanced": "configs/vae_balanced.yaml",
    "baseline_mixed": "configs/vae_balanced_mixed.yaml",
}

assert RUN_PROFILE in PROFILE_TO_CONFIG, f"Unknown RUN_PROFILE: {RUN_PROFILE}"
CONFIG = PROFILE_TO_CONFIG[RUN_PROFILE]

# Current loaders expect one dataset root.
DATA_DIR = DATASET_DIR

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Profile: {RUN_PROFILE}")
print(f"Data:    {DATA_DIR}")
print(f"Output:  {OUTPUT_DIR}")
print(f"Config:  {CONFIG}")


In [ ]:
# 5. Run training
!python scripts/train.py --config $CONFIG --data_dir $DATA_DIR --output_dir $OUTPUT_DIR

In [ ]:
# 6. Load frozen VAE checkpoint and sanity-check training outputs
import os, sys
from pathlib import Path

REPO_DIR = "/content/thesis_hapticAE"
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import numpy as np
import torch

from src.utils.config import load_config
from src.utils.seed import set_seed
from src.data.loaders import build_dataloaders, build_model, load_checkpoint
from src.eval.visualize import plot_loss_curves

config = load_config(CONFIG)
set_seed(config['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data_cfg = config['data']

run_name = config.get('run_name', 'vae_default')
run_dir = Path(OUTPUT_DIR) / run_name
ckpt_path = run_dir / 'best_model.pt'
metrics_path = run_dir / 'metrics.npz'
assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path}"

model = build_model(config, device)
load_checkpoint(model, str(ckpt_path), device)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

print('Frozen VAE checkpoint:', ckpt_path)
print('Metrics:', metrics_path, 'exists=', metrics_path.exists())
print('Trainable parameters after freeze:', sum(p.numel() for p in model.parameters() if p.requires_grad))

if metrics_path.exists():
    metrics = np.load(metrics_path)
    plot_loss_curves(metrics['train_losses'].tolist(), metrics['val_losses'].tolist())

---
## PCA and Basic Controls

Minimal PCA improvement included here: PCA is fitted only on deterministic VAE `mu`, and the notebook saves latent sample ids plus PCA artifacts for reuse.

In [ ]:
# 7. Extract deterministic mu latents, fit PCA, and save reusable artifacts
import csv
import json
import pickle

from src.pipelines.latent_extraction import extract_latent_vectors
from src.pipelines.pca_control import fit_pca_pipeline

PCA_DIR = Path(OUTPUT_DIR) / 'pca'
PCA_DIR.mkdir(parents=True, exist_ok=True)
N_COMPONENTS = 8

pca_data = build_dataloaders(config, DATA_DIR, batch_size=64, full_dataset=True)
sample_ids = [os.path.relpath(p, DATA_DIR).replace(os.sep, '/') for p in pca_data['wav_files']]

# Deterministic mu is the default PCA input. Do not use sampled z for this first frozen workflow.
Z_mu = extract_latent_vectors(model, pca_data['all_loader'], device).astype(np.float32)
np.save(PCA_DIR / 'Z_mu.npy', Z_mu)
np.save(PCA_DIR / 'Z.npy', Z_mu)  # Compatibility with existing scripts that expect Z.npy.

with open(PCA_DIR / 'sample_ids.json', 'w') as f:
    json.dump(sample_ids, f, indent=2)
with open(PCA_DIR / 'sample_ids.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['row_index', 'sample_id'])
    writer.writerows([[i, sid] for i, sid in enumerate(sample_ids)])

pipe, Z_pca = fit_pca_pipeline(Z_mu, n_components=N_COMPONENTS, save_dir=str(PCA_DIR))
pca = pipe.named_steps['pca']
scaler = pipe.named_steps['scaler']

ranges = []
for axis in range(Z_pca.shape[1]):
    col = Z_pca[:, axis]
    lo, hi = np.percentile(col, [5, 95])
    ranges.append({
        'axis': axis,
        'pc': f'PC{axis + 1}',
        'p5': float(lo),
        'p95': float(hi),
        'min': float(col.min()),
        'max': float(col.max()),
        'mean': float(col.mean()),
        'std': float(col.std()),
    })

np.savez(
    PCA_DIR / 'pca_artifacts.npz',
    components=pca.components_.astype(np.float32),
    explained_variance=pca.explained_variance_.astype(np.float32),
    explained_variance_ratio=pca.explained_variance_ratio_.astype(np.float32),
    pca_mean=pca.mean_.astype(np.float32),
    scaler_mean=scaler.mean_.astype(np.float32),
    scaler_scale=scaler.scale_.astype(np.float32),
    Z_pca=Z_pca.astype(np.float32),
)
with open(PCA_DIR / 'pca_ranges.json', 'w') as f:
    json.dump(ranges, f, indent=2)

print('Saved PCA directory:', PCA_DIR)
print('Latents:', Z_mu.shape)
print('PCA scores:', Z_pca.shape)
print('Total explained variance:', float(pca.explained_variance_ratio_.sum()))

In [ ]:
# 8. Build basic controllable PC specification
from src.pipelines.control_spec import build_controls_spec, save_controls_spec, build_controls_table_md

CTRL_DIR = Path(OUTPUT_DIR) / 'controls'
CTRL_DIR.mkdir(parents=True, exist_ok=True)

spec = build_controls_spec(
    pipe,
    model,
    device,
    Z_pca,
    pca.explained_variance_ratio_,
    T=data_cfg['T'],
    sr=data_cfg['sr'],
    n_sweep_steps=11,
)
save_controls_spec(spec, str(CTRL_DIR / 'controls_spec.json'))

controls_table = build_controls_table_md(spec)
with open(CTRL_DIR / 'controls_table.md', 'w', encoding='utf-8') as f:
    f.write(controls_table)

print('Saved controls spec:', CTRL_DIR / 'controls_spec.json')
print('Saved controls table:', CTRL_DIR / 'controls_table.md')
print('\n'.join(controls_table.splitlines()[:20]))

---
## Freeze to Google Drive

This is the stable handoff for later Frozen VAE + PC labeling + LLM control notebooks.

In [ ]:
# 9. Copy VAE checkpoint, PCA artifacts, and basic controls to Google Drive
from google.colab import drive
from pathlib import Path
import hashlib
import json
import shutil
import time

drive.mount('/content/drive')

FROZEN_ROOT = Path('/content/drive/MyDrive/thesis/frozen_model_outputs')
FROZEN_RUN_DIR = FROZEN_ROOT / run_name
FROZEN_RUN_DIR.mkdir(parents=True, exist_ok=True)

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def copy_file(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    return {'path': str(dst), 'sha256': file_sha256(dst)}

files = {}
files['checkpoint'] = copy_file(ckpt_path, FROZEN_RUN_DIR / 'best_model.pt')
if metrics_path.exists():
    files['metrics'] = copy_file(metrics_path, FROZEN_RUN_DIR / 'metrics.npz')
files['config'] = copy_file(Path(CONFIG), FROZEN_RUN_DIR / 'config.yaml')

frozen_pca_dir = FROZEN_RUN_DIR / 'pca'
frozen_controls_dir = FROZEN_RUN_DIR / 'controls'
shutil.copytree(PCA_DIR, frozen_pca_dir, dirs_exist_ok=True)
shutil.copytree(CTRL_DIR, frozen_controls_dir, dirs_exist_ok=True)

manifest = {
    'created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    'run_name': run_name,
    'source_output_dir': str(OUTPUT_DIR),
    'frozen_run_dir': str(FROZEN_RUN_DIR),
    'checkpoint_path': str(FROZEN_RUN_DIR / 'best_model.pt'),
    'config_path': str(FROZEN_RUN_DIR / 'config.yaml'),
    'metrics_path': str(FROZEN_RUN_DIR / 'metrics.npz') if (FROZEN_RUN_DIR / 'metrics.npz').exists() else None,
    'pca_dir': str(frozen_pca_dir),
    'pca_pipe_path': str(frozen_pca_dir / 'pca_pipe.pkl'),
    'pca_artifacts_path': str(frozen_pca_dir / 'pca_artifacts.npz'),
    'latent_path': str(frozen_pca_dir / 'Z_mu.npy'),
    'sample_ids_path': str(frozen_pca_dir / 'sample_ids.json'),
    'controls_dir': str(frozen_controls_dir),
    'controls_spec_path': str(frozen_controls_dir / 'controls_spec.json'),
    'controls_table_path': str(frozen_controls_dir / 'controls_table.md'),
    'files': files,
}

with open(FROZEN_RUN_DIR / 'frozen_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
with open(FROZEN_ROOT / 'latest_frozen_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print('Frozen run directory:', FROZEN_RUN_DIR)
print('Manifest:', FROZEN_RUN_DIR / 'frozen_manifest.json')
print('Latest manifest:', FROZEN_ROOT / 'latest_frozen_manifest.json')